# Case Study: Sleep Deprivation Study (GAMM)

**Duration:** 18 min | **Level:** ⭐⭐⭐

Analyze classic sleep deprivation data with random slopes.

In [ ]:
import sys
sys.path.append('../')
from utils import load_sleepstudy_data
import numpy as np
from aurora.models.gamm import fit_gamm

df = load_sleepstudy_data()
print(f'Loaded {len(df)} sleep measurements')
print(f'Subjects: {df["subject"].nunique()}')
print(df.head())

In [ ]:
# GAMM with random intercepts + slopes
n = len(df)
subjects = df['subject'].unique()
n_subj = len(subjects)
subj_map = {s: i for i, s in enumerate(subjects)}

X = np.column_stack([np.ones(n), df['days']])
y = df['reaction'].values

# Z: random intercepts + slopes
Z = np.zeros((n, 2*n_subj))
for i, (subj, days) in enumerate(zip(df['subject'], df['days'])):
    idx = subj_map[subj]
    Z[i, idx] = 1  # Random intercept
    Z[i, n_subj + idx] = days  # Random slope

result = fit_gamm(X, y, Z, family='gaussian', method='pql')
print(f'\nGAMM Results:')
print(f'Fixed intercept: {result.fixed_effects[0]:.1f} ms')
print(f'Fixed slope: {result.fixed_effects[1]:.1f} ms/day')
print(f'Random effects SD: {np.sqrt(result.random_effects_variance):.1f}')

In [ ]:
# Visualize subject trajectories
import matplotlib.pyplot as plt

pred = result.predict(X, Z)
df['pred'] = pred

plt.figure(figsize=(12, 6))
for subj in subjects[:5]:  # First 5 subjects
    subj_data = df[df['subject'] == subj]
    plt.plot(subj_data['days'], subj_data['reaction'], 'o-', alpha=0.6, label=f'Subject {subj}')
    plt.plot(subj_data['days'], subj_data['pred'], '--', alpha=0.8)

plt.xlabel('Days of sleep deprivation')
plt.ylabel('Reaction time (ms)')
plt.title('Sleep Study: Individual Trajectories')
plt.legend()
plt.tight_layout()
plt.show()

## Summary

✅ Random slopes capture individual variation  
✅ ~10 ms/day average deterioration  
✅ Large inter-individual differences  

---
**Author:** Lucy E. Arias